# Task 1

In [30]:
import pandas as pd
import numpy as np

def answer_one():
    # --- Energy ---
    # 17 rows to skip at the beginning, 37 rows to skip at the end
    energy = pd.read_excel('resource/Energy Indicators.xls', skiprows=17, skipfooter=37)

    energy = energy.iloc[:, 2:]

    energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable']

    energy = energy.replace('...', np.nan)

    energy['Energy Supply'] = energy['Energy Supply'] * 1000000

    # Dictionary to rename the countries
    di = {
        "Republic of Korea": "South Korea",
        "United States of America": "United States",
        "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
        "China, Hong Kong Special Administrative Region": "Hong Kong"
    }

    # Remove digits from the 'Country' column
    energy['Country'] = energy['Country'].str.replace(r'\d+', '', regex=True)
    # Remove parentheses and their contents from the 'Country' column
    energy['Country'] = energy['Country'].str.replace(r' \(.*\)', '', regex=True)
    #Remove leading and trailing whitespace from the 'Country' column
    energy['Country'] = energy['Country'].str.strip()


    energy['Country'] = energy['Country'].replace(di)

    # --- GDP ---
    GDP = pd.read_csv('resource/world_bank.csv', skiprows=3)

    # Dictionary to rename the countries
    gdp_rename = {
        "Korea, Rep.": "South Korea",
        "Iran, Islamic Rep.": "Iran",
        "Hong Kong SAR, China": "Hong Kong"
    }

    GDP['Country Name'] = GDP['Country Name'].replace(gdp_rename)

    # selecting only the columns we need
    columns_to_keep = ['Country Name', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015']
    GDP = GDP[columns_to_keep]

    # --- ScimEn ---
    ScimEn = pd.read_excel('resource/scimagojr.xlsx')
    ScimEn_top15 = ScimEn[:15]

    # Merging the dataframes
    merged = pd.merge(ScimEn_top15, energy, how='inner', on='Country')
        
    final = pd.merge(merged, GDP, how='inner', left_on='Country', right_on='Country Name')
        
    final = final.set_index('Country')
        
    return final

df = answer_one()
df

,Rank,Region,Documents,Citable documents,Citations,Self-citations,Citations per document,H index,Energy Supply,Energy Supply per Capita,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
Country,,,,,,,,,,,,,,,,,,,,,
China,1,Asiatic Region,273437,272374,2336764,1615239,8.55,245,127191000000,93,...,2752118542130,3550327569655,4594336785738,5101691085626,6087191720867,7551545321800,8532185615747,9570470577245,10475624783236,11061573199440
United States,2,Northern America,175891,172431,2230544,724472,12.68,363,90838000000,286,...,13815586948000,14474226905000,14769857911000,14478064934000,15048964444000,15599728123000,16253972230000,16843190993000,17550680174000,18206020741000
India,3,Asiatic Region,55082,53775,463165,162944,8.41,181,33195000000,26,...,940259888792,1216736448906,1198895147695,1341888016989,1675615502766,1823051829895,1827637579585,1856721494835,2039126469963,2103588347242
Japan,4,Asiatic Region,50523,50065,488062,119930,9.66,193,18984000000,149,...,4601662661030,4579749785985,5106679413138,5289493734900,5759071769013,6233147172341,6272362996105,5212328181166,4896994405353,4444930651964
United Kingdom,5,Western Europe,43389,42284,615670,111290,14.19,226,7920000000,124,...,2709978165671,3092996468387,2931683721187,2417565709974,2491397494468,2666403005061,2706340967031,2786315215250,3065223279584,2934857946213
Germany,6,Western Europe,38739,38013,433148,95145,11.18,196,13261000000,165,...,2994703642024,3425578382922,3745264093617,3411261212652,3399667820000,3749314991051,3527143188785,3733804649549,3889093051023,3357585719352
Russian Federation,7,Eastern Europe,36735,36560,115938,54993,3.16,90,30709000000,214,...,989932059221,1299703459800,1660848058303,1222645887209,1524916698233,2045922727570,2208293528674,2292470104248,2059241582146,1363482179761
Canada,8,Northern America,33472,32863,568080,100953,16.97,227,10431000000,296,...,1319264809591,1468820407783,1552989690722,1374625142157,1617343367486,1793326630175,1828366481522,1846597421835,1805749878440,1556508816217
Italy,9,Western Europe,27983,26940,352993,87828,12.61,166,6530000000,109,...,1949551719390,2213102482751,2408655348719,2199928804119,2136099955237,2294994296590,2086957656822,2141924094299,2162009615997,1836637711061


# Task 2

In [17]:
def answer_two():
    Top15 = answer_one()

    gdp_columns = Top15.loc[:, '2006':'2015']
    
    gdp_columns = gdp_columns.apply(pd.to_numeric, errors='coerce')

    avgGDP = gdp_columns.mean(axis=1)

    avgGDP = avgGDP.sort_values(ascending=False)

    avgGDP.name = 'avgGDP'
    
    return avgGDP

answer_two()

Country
United States         1.570403e+13
China                 6.927707e+12
Japan                 5.239642e+12
Germany               3.523342e+12
United Kingdom        2.780276e+12
France                2.691337e+12
Italy                 2.142986e+12
Brazil                1.988889e+12
Russian Federation    1.666746e+12
Canada                1.616359e+12
India                 1.602352e+12
Spain                 1.400886e+12
South Korea           1.221372e+12
Australia             1.207513e+12
Iran                  4.563261e+11
Name: avgGDP, dtype: float64

# Task 3

In [29]:
def answer_three():
    
    TargetCountry = answer_two().index[5]

    Data = answer_one().loc[TargetCountry, ['2006', '2015']]

    gdp_data_earlier = Data.iloc[0]
    gdp_data_later = Data.iloc[1]

    gdp_data_earlier = float(gdp_data_earlier)
    gdp_data_later = float(gdp_data_later)

    gdp_change = gdp_data_later - gdp_data_earlier

    return gdp_change

answer_three()

118652421858.0

# Task 4

In [38]:
def answer_four():
    Top15 = answer_one()

    ratio = Top15['Self-citations'] / Top15['Citations']

    ratio.sort_values(ascending=False, inplace=True)

    country_name = ratio.index[0]
    
    max_ratio_value = ratio.iloc[0]
    
    return (country_name, max_ratio_value)


answer_four()

('China', np.float64(0.6912289816173135))

# Task 5

In [51]:
def answer_five():
    
    Top15 = answer_one()

    Population = Top15['Energy Supply'] / Top15['Energy Supply per Capita']

    Population.sort_values(ascending=False, inplace=True)

    return Population

Population = answer_five()

country_name = Population.index[2]
pop_count = int(Population.iloc[2])

# You wont streeng? Okay)   
result = f"Country {country_name} have population {pop_count} people"    

print(result)


Country United States have population 317615384 people


# Task 6

In [57]:
def answer_six():
    Top15 = answer_one()

    Population = answer_five()

    docPerPeople = Top15['Citable documents'] / Population

    return docPerPeople.corr(Population)

answer_six()
# Hmm, interesting, more people - less documents per person

np.float64(-0.5671338323842716)

# Task 7

In [ ]:
def answer_seven():
    Top15 = answer_one()
    
    ContinentDict  = {'China':'Asia', 
                      'United States':'North America', 
                      'Japan':'Asia', 
                      'United Kingdom':'Europe', 
                      'Russian Federation':'Europe', 
                      'Canada':'North America', 
                      'Germany':'Europe', 
                      'India':'Asia',
                      'France':'Europe', 
                      'South Korea':'Asia', 
                      'Italy':'Europe', 
                      'Spain':'Europe', 
                      'Iran':'Asia',
                      'Australia':'Australia', 
                      'Brazil':'South America'}
    
    Top15['Population'] = answer_five() 
    
    Top15['Continent'] = Top15.index.to_series().map(ContinentDict)

    result = Top15.groupby('Continent')['Population'].agg(['size', 'sum', 'mean', 'std'])
    
    return result

answer_seven()

,size,sum,mean,std
Continent,,,,
Asia,5,2898666386.6106,579733277.32212,6.790979e+08
Australia,1,23316017.316017,23316017.316017,NaN
Europe,6,457929667.216372,76321611.202729,3.464767e+07
North America,2,352855249.48025,176427624.740125,1.996696e+08
South America,1,205915254.237288,205915254.237288,NaN
